# Lakehouse Veri Keşfi ve İlişkiler

Bu notebook, `analytics.duckdb` içindeki verileri read-only olarak inceler.
Amaç: hangi veri ailelerinin bulunduğunu, tabloların kolonlarını, mantıksal foreign key ilişkilerini, eksiklikleri ve basit analiz örneklerini somutlaştırmak.

Not: DuckDB içinde fiziksel FOREIGN KEY constraint'leri tanımlı olmayabilir. Aşağıdaki ilişkiler kolon isimleri ve veri sözlükleri üzerinden mantıksal olarak test edilir.

Notebook lakehouse dosyasının mevcut snapshot'ını salt okunur inceler. Kayıtlı çıktılar son çalıştırmaya aittir; güncel sonuç için hücreler baştan sona yeniden çalıştırılmalıdır. Kaynak veritabanı bu notebook tarafından değiştirilmez. Konut örnekleri keşif amaçlıdır; veri evrenini veya genel agent'ın izlemesi gereken yolu tanımlamaz.

In [1]:
from pathlib import Path
import sys
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / 'data_pipeline' / 'lakehouse' / 'analytics.duckdb').exists():
        repo_root = candidate
        break
DB_PATH = repo_root / 'data_pipeline' / 'lakehouse' / 'analytics.duckdb'
if not DB_PATH.is_file():
    raise FileNotFoundError(f'Lakehouse bulunamadı: {DB_PATH}')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from agentic_analytics.lakehouse.analysis import residualize_fixed_effects

con = duckdb.connect(str(DB_PATH), read_only=True)
print(f'DuckDB: {DB_PATH.relative_to(repo_root).as_posix()}')
print('Bağlantı: read-only')

DuckDB: data_pipeline/lakehouse/analytics.duckdb
Bağlantı: read-only


## 1. Lakehouse genel görünümü
Schema, tablo ve satır sayılarını inceleyelim.

In [2]:
schemas = con.sql("""
    SELECT DISTINCT table_schema AS schema_name
    FROM information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY schema_name
""").df()
tables = con.sql("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_schema, table_name
""").df()
tables['qualified_name'] = tables['table_schema'] + '.' + tables['table_name']
tables['row_count'] = tables['qualified_name'].map(lambda name: con.sql(f'SELECT count(*) AS n FROM {name}').fetchone()[0])
print(f'Schema sayısı: {len(schemas)} | Tablo sayısı: {len(tables)}')
display(schemas)
display(tables[['qualified_name', 'row_count']])

Schema sayısı: 10 | Tablo sayısı: 64


,schema_name
0,analysis
1,bddk
2,catalog
3,evds
4,evidence
5,quality
6,regional
7,risk_center
8,tbb
9,tuik


,qualified_name,row_count
0,analysis.housing_credit_monthly,66
1,analysis.housing_credit_quarterly,22
2,bddk.finturk_branch_zero_fallback_audit,1328
3,bddk.finturk_measurements,936512
4,bddk.finturk_metric_dictionary,76
...,...,...
59,tbb.reporting_banks,679
60,tbb.source_gaps,1
61,tuik.province_housing_sales_evds_reconciliation,25272
62,tuik.province_housing_sales_identity_zero_fall...,10


In [3]:
storage = con.sql('PRAGMA database_size').df()
display(storage)
display(tables.groupby('table_schema', as_index=False)['row_count'].sum().sort_values('row_count', ascending=False))

,database_name,database_size,block_size,total_blocks,used_blocks,free_blocks,wal_size,memory_usage,memory_limit
0,analytics,99.5 MiB,262144,398,398,0,0 bytes,7.7 MiB,25.5 GiB


,table_schema,row_count
1,bddk,3319323
3,evds,149236
9,tuik,56872
2,catalog,55635
6,regional,1896
8,tbb,953
7,risk_center,486
0,analysis,88
5,quality,22
4,evidence,5


## 2. Kolonlar, tipler ve NULL oranları
Her tablo için kolon adlarını ve veri tiplerini gösteriyoruz. Büyük tablolar için NULL oranı ayrıca raporlanıyor.

In [4]:
columns = con.sql("""
    SELECT table_schema, table_name, ordinal_position, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_schema, table_name, ordinal_position
""").df()
for qualified_name, group in columns.groupby(['table_schema', 'table_name']):
    schema_name, table_name = qualified_name
    print(f'\n### {schema_name}.{table_name}')
    display(group[['ordinal_position', 'column_name', 'data_type']])


### analysis.housing_credit_monthly


,ordinal_position,column_name,data_type
0,1,month,VARCHAR
1,2,bank_group_name_tr,VARCHAR
2,3,metric_name_tr,VARCHAR
3,4,tp_million_tl,BIGINT
4,5,yp_million_tl,BIGINT
...,...,...,...
93,94,rate_down_real_stock_not_up_quality_screened,BOOLEAN
94,95,event_count,BIGINT
95,96,event_ids,VARCHAR
96,97,event_titles,VARCHAR



### analysis.housing_credit_quarterly


,ordinal_position,column_name,data_type
98,1,quarter,VARCHAR
99,2,bddk_housing_credit_stock_million_tl,BIGINT
100,3,finturk_domestic_provinces_million_try,DOUBLE
101,4,finturk_abroad_million_try,DOUBLE
102,5,finturk_all_geographies_million_try,DOUBLE
...,...,...,...
177,80,TP_MK_F_BILESIK,DOUBLE
178,81,TP_KKM_K1,DOUBLE
179,82,TP_KKM_K2,DOUBLE
180,83,TP_KKM_K4,DOUBLE



### bddk.finturk_branch_zero_fallback_audit


...

,ordinal_position,column_name,data_type
2188,1,province_name,VARCHAR
2189,2,province_key,VARCHAR
2190,3,month,VARCHAR
2191,4,metric_code,VARCHAR
2192,5,evds_value,DOUBLE
2193,6,direct_value,DOUBLE
2194,7,tuik_usable_value,DOUBLE
2195,8,value_origin,VARCHAR
2196,9,source_csv_sha256,VARCHAR
2197,10,difference_direct,DOUBLE



### tuik.province_housing_sales_identity_zero_fallbacks


,ordinal_position,column_name,data_type
2199,1,province_name,VARCHAR
2200,2,province_key,VARCHAR
2201,3,month,VARCHAR
2202,4,metric_code,VARCHAR
2203,5,direct_value,DOUBLE
2204,6,value,DOUBLE
2205,7,value_origin,VARCHAR
2206,8,derivation,VARCHAR
2207,9,source_row_present,BOOLEAN
2208,10,is_identity_derived,BOOLEAN



### tuik.province_housing_sales_monthly


,ordinal_position,column_name,data_type
2215,1,province_name,VARCHAR
2216,2,province_key,VARCHAR
2217,3,month,VARCHAR
2218,4,metric_code,VARCHAR
2219,5,direct_value,DOUBLE
2220,6,value,DOUBLE
2221,7,value_origin,VARCHAR
2222,8,derivation,VARCHAR
2223,9,source_row_present,BOOLEAN
2224,10,is_identity_derived,BOOLEAN


In [5]:
def null_profile(qualified_name):
    cols = columns.loc[
        (columns.table_schema + '.' + columns.table_name).eq(qualified_name), 'column_name'
    ].tolist()
    total = con.sql(f'SELECT count(*) FROM {qualified_name}').fetchone()[0]
    rows = []
    for col in cols:
        nulls = con.sql(f'SELECT count(*) FROM {qualified_name} WHERE "{col}" IS NULL').fetchone()[0]
        rows.append({'table': qualified_name, 'column': col, 'null_count': nulls, 'null_rate': nulls / total if total else 0})
    return pd.DataFrame(rows)
for target in ['analysis.housing_credit_monthly', 'analysis.housing_credit_quarterly', 'regional.housing_quarterly']:
    if target in tables.qualified_name.values:
        print(target)
        display(null_profile(target).sort_values('null_rate', ascending=False))

analysis.housing_credit_monthly


,table,column,null_count,null_rate
97,analysis.housing_credit_monthly,event_verification_levels,61,0.924242
96,analysis.housing_credit_monthly,event_titles,61,0.924242
95,analysis.housing_credit_monthly,event_ids,61,0.924242
45,analysis.housing_credit_monthly,TP_HANEBEK_HAN18A,60,0.909091
53,analysis.housing_credit_monthly,TP_PKAUO_S04_C_U,53,0.803030
...,...,...,...,...
48,analysis.housing_credit_monthly,TP_KFE_TR10,0,0.000000
47,analysis.housing_credit_monthly,TP_KFE_TR,0,0.000000
46,analysis.housing_credit_monthly,TP_IN_RH2_TOPLAM_TOP_D,0,0.000000
44,analysis.housing_credit_monthly,TP_GY1_N2_MA,0,0.000000


analysis.housing_credit_quarterly


,table,column,null_count,null_rate
50,analysis.housing_credit_quarterly,TP_HANEBEK_HAN18A,20,0.909091
58,analysis.housing_credit_quarterly,TP_PKAUO_S04_C_U,17,0.772727
82,analysis.housing_credit_quarterly,TP_KKM_K4,3,0.136364
81,analysis.housing_credit_quarterly,TP_KKM_K2,3,0.136364
80,analysis.housing_credit_quarterly,TP_KKM_K1,3,0.136364
...,...,...,...,...
31,analysis.housing_credit_quarterly,TP_BKEA_S065,0,0.000000
30,analysis.housing_credit_quarterly,TP_BKEA_S064,0,0.000000
29,analysis.housing_credit_quarterly,TP_BKEA_S063,0,0.000000
28,analysis.housing_credit_quarterly,TP_BKEA_S062,0,0.000000


regional.housing_quarterly


,table,column,null_count,null_rate
39,regional.housing_quarterly,housing_unit_price_proxy_peer_count,1620,0.909091
42,regional.housing_quarterly,housing_unit_rent_yoy_pct,506,0.283951
40,regional.housing_quarterly,housing_unit_price_yoy_pct,466,0.261504
44,regional.housing_quarterly,regional_ykke_yoy_pct,324,0.181818
41,regional.housing_quarterly,housing_unit_price_with_proxy_yoy_pct,324,0.181818
43,regional.housing_quarterly,regional_kfe_yoy_pct,324,0.181818
45,regional.housing_quarterly,housing_credit_yoy_pct,324,0.181818
17,regional.housing_quarterly,housing_unit_rent_try_per_m2,210,0.117845
16,regional.housing_quarterly,housing_unit_price_try_per_m2,162,0.090909
35,regional.housing_quarterly,housing_credit_to_savings_deposit_pct,0,0.000000


Yüksek NULL oranı otomatik olarak veri hatası demek değildir. Yayımlanmayan dönem, uygulanamaz alan ve eksik kaynak gözlemi ayrılmalıdır. `analysis_ready_source` veya `analysis_ready_with_price_proxy` belirli temel metrik kümelerinin hazırlığını anlatır; her sütunun dolu olduğunu garanti etmez. Her analiz yalnız kullandığı sütunların eksikliklerini kontrol eder. Resmî kaynak null'u, kesin bir kaynak kimliğiyle türetilmiş değer ve tahmini fiyat proxy'si aynı şey değildir.

## 3. Mantıksal foreign key ilişkileri
Bu ilişkiler fiziksel constraint değil, veri modelinin beklenen anahtarlarıdır. Orphan sayısı sıfır olmalıdır.

In [6]:
def has_table(name):
    return name in set(tables.qualified_name)

relationships = [
    ('bddk.monthly_measurements', 'bddk.monthly_metric_dictionary', ['table_no', 'metric_code'], ['table_no', 'metric_code'], 'BDDK aylık ölçüm -> metrik sözlüğü'),
    ('bddk.weekly_measurements', 'bddk.weekly_metric_dictionary', ['table_id', 'metric_code'], ['table_id', 'metric_code'], 'BDDK haftalık ölçüm -> metrik sözlüğü'),
    ('bddk.weekly_measurements', 'bddk.weekly_source_tables', ['observation_date', 'table_id', 'group_code'], ['observation_date', 'table_id', 'group_code'], 'BDDK haftalık ölçüm -> kaynak tablo'),
    ('bddk.finturk_measurements', 'bddk.finturk_metric_dictionary', ['measure_code'], ['measure_code'], 'FinTürk ölçüm -> metrik sözlüğü'),
    ('bddk.finturk_measurements', 'bddk.finturk_source_tables', ['quarter', 'table_no', 'source_file', 'source_sha256'], ['quarter', 'table_no', 'source_file', 'source_sha256'], 'FinTürk ölçüm -> kaynak tablo'),
    ('regional.housing_quarterly', 'regional.province_dimension', ['province_key'], ['province_key'], 'İl paneli -> il boyutu'),
    ('tuik.province_housing_sales_evds_reconciliation', 'tuik.province_housing_sales_monthly', ['province_key', 'month', 'metric_code'], ['province_key', 'month', 'metric_code'], 'TÜİK uzlaştırma -> TÜİK aylık veri'),
]


def orphan_count(child, parent, child_keys, parent_keys):
    if not has_table(child) or not has_table(parent):
        return None
    condition = ' AND '.join(f'c."{a}" = p."{b}"' for a, b in zip(child_keys, parent_keys))
    return con.sql(f'SELECT count(*) FROM {child} c ANTI JOIN {parent} p ON {condition}').fetchone()[0]

relationship_rows = []
for child, parent, child_keys, parent_keys, description in relationships:
    count = orphan_count(child, parent, child_keys, parent_keys)
    relationship_rows.append({'relationship': description, 'child': child, 'parent': parent, 'orphan_rows': count, 'status': 'SKIP' if count is None else ('PASS' if count == 0 else 'CHECK')})

for observation_table in sorted(name for name in tables.qualified_name if name.startswith('evds.') and name.endswith('_observations')):
    count = orphan_count(observation_table, 'evds.series_catalog', ['series_code'], ['series_code'])
    relationship_rows.append({'relationship': 'EVDS observation -> seri kataloğu', 'child': observation_table, 'parent': 'evds.series_catalog', 'orphan_rows': count, 'status': 'SKIP' if count is None else ('PASS' if count == 0 else 'CHECK')})

relationship_df = pd.DataFrame(relationship_rows)
display(relationship_df)

,relationship,child,parent,orphan_rows,status
0,BDDK aylık ölçüm -> metrik sözlüğü,bddk.monthly_measurements,bddk.monthly_metric_dictionary,0,PASS
1,BDDK haftalık ölçüm -> metrik sözlüğü,bddk.weekly_measurements,bddk.weekly_metric_dictionary,0,PASS
2,BDDK haftalık ölçüm -> kaynak tablo,bddk.weekly_measurements,bddk.weekly_source_tables,0,PASS
3,FinTürk ölçüm -> metrik sözlüğü,bddk.finturk_measurements,bddk.finturk_metric_dictionary,0,PASS
4,FinTürk ölçüm -> kaynak tablo,bddk.finturk_measurements,bddk.finturk_source_tables,0,PASS
5,İl paneli -> il boyutu,regional.housing_quarterly,regional.province_dimension,0,PASS
6,TÜİK uzlaştırma -> TÜİK aylık veri,tuik.province_housing_sales_evds_reconciliation,tuik.province_housing_sales_monthly,0,PASS
7,EVDS observation -> seri kataloğu,evds.household_finance_observations,evds.series_catalog,0,PASS
8,EVDS observation -> seri kataloğu,evds.housing_controls_observations,evds.series_catalog,0,PASS
9,EVDS observation -> seri kataloğu,evds.housing_observations,evds.series_catalog,0,PASS


## Çıktının yorumu
`status=PASS` ve `orphan_rows=0`, ölçüm tablolarındaki anahtarların ilgili sözlük/kaynak tablolarında karşılığı olduğunu gösterir. `SKIP`, tablonun mevcut olmadığını; `CHECK`, eşleşmeyen kayıt bulunduğunu anlatır. PASS sonucu verinin ekonomik olarak doğru olduğunu değil, veri modelindeki teknik bağlantının kopuk olmadığını kanıtlar.

In [7]:
display(con.sql("""
    SELECT source_system,
           count(*) AS total_metrics,
           count(*) FILTER (WHERE observation_available) AS available_metrics,
           round(100.0 * count(*) FILTER (WHERE observation_available) / NULLIF(count(*), 0), 2) AS coverage_pct
    FROM catalog.metrics
    GROUP BY source_system
    ORDER BY source_system
""").df())
display(con.sql("""
    SELECT count(*) AS total_metrics,
           count(*) FILTER (WHERE observation_available) AS queryable_metrics,
           round(100.0 * count(*) FILTER (WHERE observation_available) / NULLIF(count(*), 0), 2) AS queryable_pct
    FROM catalog.metrics
""").df())

,source_system,total_metrics,available_metrics,coverage_pct
0,BDDK_FINTURK,76,76,100.00
1,BDDK_MONTHLY,2108,2108,100.00
2,BDDK_WEEKLY,517,515,99.61
3,REGIONAL_HOUSING_ANALYSIS,33,33,100.00
4,TBB_REPORTS,60,60,100.00
5,TBB_RISK_CENTER,5,5,100.00
6,TCMB_EVDS,52696,584,1.11
7,TCMB_EVDS_DERIVED,1,1,100.00
8,TUIK_DATA_PORTAL,5,5,100.00


,total_metrics,queryable_metrics,queryable_pct
0,55501,3387,6.1


## Çıktının yorumu
`coverage_pct`, birleşik katalogda yerel gözlem varlığı işaretlenmiş metriklerin oranıdır. Bu oran veri kalitesi veya her dönemde dolu gözlem garantisi değildir; tamamen null kaynak serileri de indirilmiş olabilir. Ayrıca bu katalog, diskteki eski veya ek kaynak dosyalarının eksiksiz envanteri sayılmamalıdır. Seri ve dönem bazında gerçek dolu gözlem sayısı ayrıca incelenmelidir.

## 4. Basit analiz 
Aşağıdaki sorgular lakehouse’taki veri ailelerinin ne anlatabildiğini gösterir.

In [8]:
display(con.sql("""
    SELECT month,
           round(bddk_housing_credit_stock_million_tl, 2) AS housing_stock_million_tl,
           round(housing_credit_real_mom_pct, 2) AS real_stock_mom_pct,
           round(TP_KTF12, 2) AS housing_rate_pct,
           round(mortgaged_sales_share_pct, 2) AS mortgaged_sales_share_pct
    FROM analysis.housing_credit_monthly
    ORDER BY month DESC
    LIMIT 12
""").df())

,month,housing_stock_million_tl,real_stock_mom_pct,housing_rate_pct,mortgaged_sales_share_pct
0,2026-06,801438,1.37,40.34,20.00
1,2026-05,782918,0.31,38.43,21.17
2,2026-04,767314,-0.74,36.62,20.32
3,2026-03,741972,1.69,34.66,22.91
4,2026-02,715799,0.56,35.16,20.10
5,2026-01,691343,-2.88,36.24,18.18
6,2025-12,678970,2.16,37.29,11.54
7,2025-11,658770,1.71,37.76,15.32
8,2025-10,642144,0.39,37.94,14.44
9,2025-09,623765,-0.73,38.80,14.19


## Çıktının yorumu
Bu çıktı son 12 ayda konut kredisi stokunu, reel aylık değişimi, kredi faizini ve ipotekli satış payını yan yana gösterir. Reel stok artmıyor ve faiz değişimi negatifse bu yalnızca dikkat çekici bir dönemdir; kredi talebi, arz koşulları, enflasyon ve banka kapsamı kontrol edilmeden faiz düşüşünün etkisi olarak yorumlanamaz.

In [9]:
display(con.sql("""
    SELECT quarter,
           round(bddk_housing_credit_stock_million_tl, 2) AS bddk_stock,
           round(finturk_domestic_provinces_million_try, 2) AS finturk_domestic_stock,
           round(balance_amount_million_try, 2) AS tbb_balance,
           round(disbursement_amount_million_try, 2) AS tbb_disbursement
    FROM analysis.housing_credit_quarterly
    ORDER BY quarter DESC
    LIMIT 8
""").df())

,quarter,bddk_stock,finturk_domestic_stock,tbb_balance,tbb_disbursement
0,2026-06,801438,801440.79,NaN,NaN
1,2026-03,741972,741975.63,675178.32,99520.74
2,2025-12,678970,678974.58,614238.89,86982.71
3,2025-09,623765,623770.00,562890.17,68307.00
4,2025-06,583265,583270.06,525421.65,58246.68
5,2025-03,548385,548390.60,495614.44,55340.64
6,2024-12,512259,512264.82,464901.93,67391.65
7,2024-09,465313,465320.39,420965.71,36915.69


## Çıktının yorumu
BDDK, FinTürk ve TBB satırlarını doğrudan eşit kabul etmeyin. BDDK sektör stokunu, FinTürk il toplamını, TBB ise raporlayan bankaların bakiyesini ve gerçek kullandırım akımını gösterir. TBB’nin 2026-06 değeri kaynakta yoksa NULL kalması beklenir; bu bir sıfır veya tahmin değildir.

In [10]:
display(con.sql("""
    SELECT province_name, quarter,
           housing_sales_total_count,
           housing_sales_mortgaged_count,
           round(mortgaged_sales_share_pct, 2) AS mortgaged_share_pct,
           analysis_ready
    FROM regional.housing_quarterly
    WHERE quarter = (SELECT max(quarter) FROM regional.housing_quarterly)
    ORDER BY housing_sales_total_count DESC
    LIMIT 15
""").df())

,province_name,quarter,housing_sales_total_count,housing_sales_mortgaged_count,mortgaged_share_pct,analysis_ready
0,İSTANBUL,2026Q2,66133.0,15328.0,23.18,True
1,ANKARA,2026Q2,31169.0,7948.0,25.50,True
2,İZMİR,2026Q2,20704.0,4558.0,22.02,True
3,ANTALYA,2026Q2,17569.0,2655.0,15.11,True
4,MERSİN,2026Q2,11933.0,2055.0,17.22,True
5,BURSA,2026Q2,11842.0,2817.0,23.79,True
6,KOCAELİ,2026Q2,9872.0,2662.0,26.97,True
7,GAZİANTEP,2026Q2,9715.0,1896.0,19.52,True
8,TEKİRDAĞ,2026Q2,9010.0,2270.0,25.19,True
9,KONYA,2026Q2,8040.0,1401.0,17.43,True


## Çıktının yorumu
Sıralama mevcut panelin son çeyreğinde toplam konut satışı yüksek olan illeri gösterir. `mortgaged_share_pct` ipotekli satışların toplam satış içindeki payıdır. `analysis_ready` seçilmiş temel kaynak kümesinin bayrağıdır; satışlar dolu olsa bile kullanılmayan fiyat alanı eksikse false olabilir. Büyük satış hacmi tek başına yüksek kredi yoğunluğu veya yüksek fiyat anlamına gelmez.

In [11]:
display(con.sql("""
    SELECT reconciliation_status, count(*) AS observation_count
    FROM tuik.province_housing_sales_evds_reconciliation
    GROUP BY reconciliation_status
    ORDER BY reconciliation_status
""").df())
weekly_source_duplicates = con.sql("""
    SELECT count(*) FROM (
        SELECT observation_date, table_id, group_code
        FROM bddk.weekly_source_tables
        GROUP BY observation_date, table_id, group_code
        HAVING count(*) > 1
    )
""").fetchone()[0]
if weekly_source_duplicates:
    raise ValueError('Haftalık kaynak anahtarları tekil değil; join ölçümleri çoğaltabilir.')
weekly_group_coverage = con.sql("""
    SELECT m.group_code, s.group_name,
           count(*) AS measurement_rows,
           count(m.value) AS non_null_rows,
           count(*) FILTER (WHERE m.value IS NULL) AS null_rows,
           count(DISTINCT m.observation_date) AS observed_weeks,
           count(*) FILTER (WHERE s.group_name IS NULL) AS missing_source_rows
    FROM bddk.weekly_measurements m
    LEFT JOIN bddk.weekly_source_tables s
      ON s.observation_date = m.observation_date
     AND s.table_id = m.table_id
     AND s.group_code = m.group_code
    GROUP BY m.group_code, s.group_name
    ORDER BY m.group_code, s.group_name
""").df()
display(weekly_group_coverage)

,reconciliation_status,observation_count
0,evds_null_tuik_identity_zero,10
1,exact_match,25262


,group_code,group_name,measurement_rows,non_null_rows,null_rows,observed_weeks,missing_source_rows
0,10001,Sektor,147154,146640,514,286,0
1,10002,Mevduat,146470,146184,286,286,0
2,10003,Kalkinma ve Yatirim,146470,146184,286,286,0
3,10004,Katilim,146470,146184,286,286,0
4,10005,Kamu,146470,146184,286,286,0
5,10006,Yabanci,146470,146184,286,286,0
6,10007,Yerli Ozel,146470,146184,286,286,0


## Çıktının yorumu
TÜİK-EVDS sonuçları doğrudan ortak değer eşleşmesi ile kaynak kimliğinden türetilmiş sıfırı ayrı gösterir. Değer uyuşmazlığında önce tarih, il, metrik ve snapshot kontrol edilmelidir. Banka tablosu ölçüm, dolu/null kayıt ve gözlenen hafta sayılarını gösterir. Farklı ekonomik metrikler toplanmaz veya ortalamaları alınmaz. Kaynak join'inin tekilliği kontrol edilir; kaynağı bulunamayan ölçümler varsa ayrıca sayılır.

## 5. Bu verilerle ne yapılabilir?

- BDDK'nın kredi, mevduat, sermaye, kârlılık ve sektör tabloları kaynak kapsamı korunarak incelenebilir.
- Aynı metrik ve banka grubu anlamı seçilerek haftalık/aylık karşılaştırmalar yapılabilir.
- İl bazında satış, fiyat, kredi ve mevduat göstergeleri karşılaştırılabilir.
- Ortak TÜİK-EVDS gözlemleri çapraz kaynak doğrulaması için kullanılabilir.
- TBB kullandırım akımı ile BDDK/FinTürk stokları farklı ölçüler olarak ele alınır.

## Dikkat edilmesi gerekenler

- Kaynakta yayımlanmayan dönemler sıfır veya tahmin değildir.
- Resmî fiyat null'ları korunur; ayrı fiyat proxy'si ancak açık seçimle kullanılabilir.
- Katalog metadatası ile indirilmiş ve sayısal gözlemi bulunan seri farklı kapsamlardır.
- Korelasyon birlikte hareketi özetler; tek başına nedensellik kanıtı değildir.
- Hazır `analysis` tabloları ham kaynak evreninin belirli birleşimleridir.

## Sonraki keşif örnekleri

1. İl bazında satış payı ve kredi yoğunluğu segmentleri.
2. Banka grubu bazında haftalık veri kapsamı.
3. Faiz değişimi ve sonraki reel kredi stoku değişiminin korelasyonu.
4. Kullanılan değişkenlere göre seçilmiş örneklemde keşifsel panel regresyonu.
5. Rolling-MAD ile olağandışı değer taraması; kalıcı değişim noktası analizi ayrı bir çalışmadır.

## il kümeleri
İl bazında satış payı ve kredi yoğunluğunu son çeyrek için karşılaştırıyoruz. `qcut` ile eşit gözlem sayısına yakın düşük/orta/yüksek kümeleri oluşturulur; bu bir segmentasyon özetidir, makine öğrenmesi sonucu değildir.

In [12]:
regional = con.sql("""
    SELECT province_name, quarter,
           housing_sales_total_count,
           mortgaged_sales_share_pct,
           housing_credit_per_capita_try,
           regional_kfe_index,
           regional_ykke_index
    FROM regional.housing_quarterly
    WHERE quarter = (SELECT max(quarter) FROM regional.housing_quarterly)
""").df()
regional_source_rows = len(regional)
regional_required = ['housing_sales_total_count', 'mortgaged_sales_share_pct', 'housing_credit_per_capita_try']
regional = regional.dropna(subset=regional_required).copy()
if regional.empty:
    raise ValueError('Segmentasyon için kullanılan sütunlarda ortak dolu gözlem bulunamadı.')
for column in ['mortgaged_sales_share_pct', 'housing_credit_per_capita_try']:
    # Tied values may collapse quantile boundaries; do not assign three labels
    # when fewer than three distinct bins exist.
    bins = pd.qcut(regional[column], q=3, labels=False, duplicates='drop')
    bin_count = bins.nunique()
    if bin_count <= 1:
        regional[f'{column}_cluster'] = 'tek düzey'
    else:
        labels = ['düşük', 'yüksek'] if bin_count == 2 else ['düşük', 'orta', 'yüksek']
        regional[f'{column}_cluster'] = bins.map(dict(enumerate(labels)))
regional['sales_rank'] = regional['housing_sales_total_count'].rank(ascending=False, method='min').astype('Int64')
print(f"Segmentasyon dönemi: {regional['quarter'].iloc[0]} | İl sayısı: {len(regional)} | Kullanılan sütunlarda eksik: {regional_source_rows - len(regional)}")
display(regional.sort_values(['housing_credit_per_capita_try', 'housing_sales_total_count'], ascending=False).head(20))
display(
    regional.groupby(['mortgaged_sales_share_pct_cluster', 'housing_credit_per_capita_try_cluster'], observed=True)
    .agg(province_count=('province_name', 'count'), average_sales=('housing_sales_total_count', 'mean'), average_credit_per_capita=('housing_credit_per_capita_try', 'mean'))
    .reset_index()
)

Segmentasyon dönemi: 2026Q2 | İl sayısı: 81 | Kullanılan sütunlarda eksik: 0


,province_name,quarter,housing_sales_total_count,mortgaged_sales_share_pct,housing_credit_per_capita_try,regional_kfe_index,regional_ykke_index,mortgaged_sales_share_pct_cluster,housing_credit_per_capita_try_cluster,sales_rank
6,ANKARA,2026Q2,31169.0,25.499695,18430.067383,255.69,352.04,yüksek,yüksek,2
31,ESKİŞEHİR,2026Q2,5690.0,22.179262,15945.319583,227.71,309.47,yüksek,yüksek,18
39,İSTANBUL,2026Q2,66133.0,23.177536,14496.807320,216.62,304.25,yüksek,yüksek,1
72,TEKİRDAĞ,2026Q2,9010.0,25.194229,13858.408463,216.84,312.50,yüksek,yüksek,9
40,İZMİR,2026Q2,20704.0,22.015070,13314.289032,225.16,323.51,yüksek,yüksek,3
51,KOCAELİ,2026Q2,9872.0,26.965154,11736.393832,235.64,332.72,yüksek,yüksek,7
24,DENİZLİ,2026Q2,4150.0,24.433735,11435.100715,222.22,315.63,yüksek,yüksek,22
21,ÇANAKKALE,2026Q2,2878.0,16.817234,10934.054724,238.31,331.06,düşük,yüksek,26
7,ANTALYA,2026Q2,17569.0,15.111845,9967.780264,194.13,255.05,düşük,yüksek,4
48,KIRIKKALE,2026Q2,1299.0,25.404157,9804.009462,254.08,359.33,yüksek,yüksek,52


,mortgaged_sales_share_pct_cluster,housing_credit_per_capita_try_cluster,province_count,average_sales,average_credit_per_capita
0,düşük,düşük,16,1677.187500,2860.437388
1,düşük,orta,5,830.200000,6050.324061
2,düşük,yüksek,6,6047.333333,9206.939781
3,orta,düşük,9,2631.444444,3753.953369
4,orta,orta,12,2937.000000,5967.851063
5,orta,yüksek,6,4963.833333,8678.818541
6,yüksek,düşük,2,1974.000000,3641.826609
7,yüksek,orta,10,2063.800000,6291.181113
8,yüksek,yüksek,15,11303.600000,11464.499036


## Çıktının yorumu
Bu çıktı illeri son çeyrekteki iki boyuta göre göreli segmentlere ayırır. Örneklem yalnız satış, ipotekli pay ve kişi başı kredi alanlarının doluluğuna göre seçilir; kullanılmayan birim fiyat eksikliği il dışlama nedeni değildir. Eşit değerler nedeniyle üç ayrı quantile oluşmuyorsa mevcut segment sayısı kullanılır. Bu segmentler ekonomik eşik veya nedensel fark göstermez. Kişi başı konut kredisi, FinTürk verilerindeki iki ölçüden ters hesaplanan yaklaşık nüfus paydasını kullanır; bu payda resmî nüfus serisi değildir.

## Banka gruplarının veri kapsamı
Haftalık BDDK ölçümlerinin banka grubu, kaynak tablosu ve para boyutundaki kapsamını sayıyoruz. Kaynak değerlerin toplamı veya ortalaması hesaplanmaz; farklı tabloların kalemleri birbirine eklenmez.

In [13]:
bank_groups = con.sql("""
    SELECT m.group_code, s.group_name, m.table_id,
           s.table_name, m.currency_dimension,
           count(*) AS measurement_rows,
           count(m.value) AS non_null_rows,
           count(*) FILTER (WHERE m.value IS NULL) AS null_rows,
           count(DISTINCT m.metric_code) AS metric_count,
           count(DISTINCT m.observation_date) AS observed_weeks,
           min(m.observation_date) AS first_week,
           max(m.observation_date) AS last_week
    FROM bddk.weekly_measurements m
    LEFT JOIN bddk.weekly_source_tables s
      ON s.observation_date = m.observation_date
     AND s.table_id = m.table_id
     AND s.group_code = m.group_code
    GROUP BY m.group_code, s.group_name, m.table_id, s.table_name, m.currency_dimension
    ORDER BY m.group_code, m.table_id, m.currency_dimension
""").df()
display(bank_groups)
print(f"Banka grubu: {bank_groups['group_code'].nunique()} | Tablo: {bank_groups['table_id'].nunique()} | Grup/tablo/para boyutu satırı: {len(bank_groups)}")
print('Ekonomik karşılaştırma için aynı table_id, metric_code, currency_dimension ve dönem seçilmelidir.')

,group_code,group_name,table_id,table_name,currency_dimension,measurement_rows,non_null_rows,null_rows,metric_count,observed_weeks,first_week,last_week
0,10001,Sektor,289,Krediler,FX,6114,6114,0,22,286,2021-01-08,2026-06-26
1,10001,Sektor,289,Krediler,TOTAL,6114,6114,0,22,286,2021-01-08,2026-06-26
2,10001,Sektor,289,Krediler,TRY,6114,6114,0,22,286,2021-01-08,2026-06-26
3,10001,Sektor,290,Takipteki Alacaklar,FX,3432,3432,0,12,286,2021-01-08,2026-06-26
4,10001,Sektor,290,Takipteki Alacaklar,TOTAL,3432,3432,0,12,286,2021-01-08,2026-06-26
...,...,...,...,...,...,...,...,...,...,...,...,...
170,10007,Yerli Ozel,295,Bankalarda Saklanan Menkul Degerler - 1,TRY,12870,12870,0,45,286,2021-01-08,2026-06-26
171,10007,Yerli Ozel,296,Bankalarda Saklanan Menkul Degerler - 2,FX,12870,12870,0,45,286,2021-01-08,2026-06-26
172,10007,Yerli Ozel,296,Bankalarda Saklanan Menkul Degerler - 2,TOTAL,12870,12870,0,45,286,2021-01-08,2026-06-26
173,10007,Yerli Ozel,296,Bankalarda Saklanan Menkul Degerler - 2,TRY,12870,12870,0,45,286,2021-01-08,2026-06-26


Banka grubu: 7 | Tablo: 9 | Grup/tablo/para boyutu satırı: 175
Ekonomik karşılaştırma için aynı table_id, metric_code, currency_dimension ve dönem seçilmelidir.


## Çıktının yorumu
`measurement_rows` ve `non_null_rows` veri kapsamını, `observed_weeks` temsil edilen hafta sayısını gösterir. Bunlar bilanço büyüklüğü veya performans sıralaması değildir. Gruplar birbirini dışlayan tek bir sınıflama oluşturmaz: sektör toplamı, banka fonksiyonu ve sahiplik grupları üst üste toplanamaz. Grup adları bu haftalık kaynağın sözlüğünden gelir; aynı sayısal kodun aylık veya FinTürk kaynağında aynı grubu temsil ettiği varsayılmaz.

## faiz sonrası gecikmeli tepki
Faiz değişimi ile reel konut kredisi stok değişiminin 1, 3 ve 6 aylık gecikmeli birlikte hareketi hesaplanır. Bu korelasyon nedensellik değildir; yalnızca ileri analiz için aday gecikmeleri gösterir.

In [14]:
monthly = con.sql("""
    SELECT month, housing_rate_change_pp, housing_credit_real_mom_pct
    FROM analysis.housing_credit_monthly
    ORDER BY month
""").df()
monthly['month'] = pd.to_datetime(monthly['month'] + '-01')
if monthly['month'].duplicated().any():
    raise ValueError('Aylık korelasyon anahtarı tekil değil.')
# Missing calendar months remain missing so shift counts months, not rows.
monthly = monthly.set_index('month').asfreq('MS')
lag_rows = []
for lag in [1, 3, 6]:
    frame = monthly.copy()
    frame['future_real_stock_change'] = frame['housing_credit_real_mom_pct'].shift(-lag)
    frame = frame.dropna(subset=['housing_rate_change_pp', 'future_real_stock_change'])
    lag_rows.append({'lag_months': lag, 'observations': len(frame), 'correlation_rate_change_vs_future_real_stock': frame['housing_rate_change_pp'].corr(frame['future_real_stock_change'])})
lag_results = pd.DataFrame(lag_rows)
display(lag_results)
print('Yorum: Bunlar gecikmeli birlikte hareket katsayılarıdır; politika etkisi veya nedensellik tahmini değildir.')

,lag_months,observations,correlation_rate_change_vs_future_real_stock
0,1,64,-0.373916
1,3,62,-0.274657
2,6,59,-0.166307


Yorum: Bunlar gecikmeli birlikte hareket katsayılarıdır; politika etkisi veya nedensellik tahmini değildir.


## Çıktının yorumu
Katsayı pozitifse faiz değişimi ile seçilen ay sonraki reel stok değişimi aynı yönde, negatifse ters yönde hareket eder. Her gecikmenin fiilî ortak gözlem sayısı tabloda gösterilir. Eksik takvim ayları korunur; satır kaydırma eksik ayları atlamaz. Kısa örneklem, enflasyon, arz ve düzenleme gibi ortak etkenler nedeniyle bu sonuçlar yalnız keşif amaçlıdır; güven aralığı veya nedensel etki üretilmez.

## Kullanılan sütunlara göre keşifsel panel regresyonu
İl ortalamaları çıkarılarak tek yönlü sabit etki dönüşümü uygulanır. Örneklem yalnız modelin kullandığı satış, kredi, ipotekli pay ve bölgesel endekslerin doluluğuna göre seçilir. Modelde kullanılmayan il birim fiyatının eksikliği satır dışlamaz. Zaman etkileri ve istatistiksel belirsizlik bu ilk modelde hesaplanmaz; sonuç nedensel değildir.

In [15]:
panel = con.sql("""
    SELECT province_key, quarter,
           housing_sales_total_count AS sales_total,
           housing_credit_per_capita_try AS credit_per_capita,
           mortgaged_sales_share_pct AS mortgaged_share,
           regional_kfe_index AS kfe,
           regional_ykke_index AS ykke
    FROM regional.housing_quarterly
""").df()
features = ['credit_per_capita', 'mortgaged_share', 'kfe', 'ykke']
panel_source_rows = len(panel)
panel = panel.dropna(subset=['province_key', 'quarter', 'sales_total', *features]).copy()
if panel.duplicated(['province_key', 'quarter']).any():
    raise ValueError('İl-çeyrek model anahtarı tekil değil.')
model_data = residualize_fixed_effects(panel, ['sales_total', *features], ['province_key'])
X = model_data[features].to_numpy()
y = model_data['sales_total'].to_numpy()
coefficients, _, rank, _ = np.linalg.lstsq(X, y, rcond=None)
if rank < len(features):
    raise ValueError('Sabit etkiler çıkarıldıktan sonra model değişkenleri bağımsız değil.')
predicted = X @ coefficients
residual = y - predicted
regression_summary = pd.DataFrame({'term': features, 'coefficient': coefficients})
print(f'Gözlem: {len(model_data)} | İl: {panel.province_key.nunique()} | Kullanılan sütunlarda eksik: {panel_source_rows - len(panel)}')
display(regression_summary)
within_total = y @ y
print(f'Within R-kare: {1 - (residual @ residual) / within_total:.4f}' if within_total > 0 else 'Within R-kare: tanımsız (sabit etki sonrası değişim yok).')
print('Katsayılar keşifseldir; standart hata, güven aralığı veya nedensel etki hesaplanmadı.')

Gözlem: 1782 | İl: 81 | Kullanılan sütunlarda eksik: 0


,term,coefficient
0,credit_per_capita,0.295202
1,mortgaged_share,-13.972670
2,kfe,-34.255979
3,ykke,20.353833


Within R-kare: 0.0376
Katsayılar keşifseldir; standart hata, güven aralığı veya nedensel etki hesaplanmadı.


## Çıktının yorumu
Within katsayıları, her ilin kendi ortalamasından sapmalarla satış hacmi arasındaki ilişkiyi özetler. Pozitif katsayı birlikte artış, negatif katsayı ters yönlü ilişki demektir. `R-kare` modelin açıklama gücünü gösterir; istatistiksel anlamlılık, standart hata ve zaman sabit etkileri hesaplanmadığı için bu sonuç karar veya nedensellik kanıtı değildir.

## Rolling-MAD ile olağandışı değer taraması
Reel kredi stokunun aylık değişimi, önceki en fazla altı takvim ayının medyanı ve medyan mutlak sapmasıyla karşılaştırılır. En az üç dolu geçmiş ay gerekir. Bu bir noktasal anomali taramasıdır; kalıcı trend veya rejim değişikliğini saptayan bir change detection modeli değildir.

In [16]:
series = con.sql("""
    SELECT month, housing_credit_real_mom_pct, housing_rate_change_pp
    FROM analysis.housing_credit_monthly
    ORDER BY month
""").df()
series['month'] = pd.to_datetime(series['month'] + '-01')
if series['month'].duplicated().any():
    raise ValueError('Anomali serisinin aylık anahtarı tekil değil.')
series = series.set_index('month').asfreq('MS')
previous = series['housing_credit_real_mom_pct'].shift(1)
window = previous.rolling(6, min_periods=3)
series['baseline_observations'] = previous.rolling(6, min_periods=1).count()
series['rolling_median'] = window.median()
series['rolling_mad'] = window.apply(lambda values: (values - values.median()).abs().median(), raw=False)
scale = 1.4826 * series['rolling_mad'].where(series['rolling_mad'].gt(0))
series['robust_z'] = (series['housing_credit_real_mom_pct'] - series['rolling_median']) / scale
series['scorable'] = series['robust_z'].notna()
series['anomaly_flag'] = series['robust_z'].abs().ge(3)
series = series.reset_index()
series['month'] = series['month'].dt.strftime('%Y-%m')
print(f"İşaretlenen dönem: {int(series['anomaly_flag'].sum())} | Skorlanamayan: {int((~series['scorable']).sum())}")
display(series.loc[series['anomaly_flag'], ['month', 'housing_credit_real_mom_pct', 'housing_rate_change_pp', 'baseline_observations', 'robust_z']])
display(series.sort_values('robust_z', key=lambda values: values.abs(), ascending=False).head(10))

İşaretlenen dönem: 7 | Skorlanamayan: 4

,month,housing_credit_real_mom_pct,housing_rate_change_pp,baseline_observations,robust_z
11,2021-12,-9.762532,0.2105,6.0,-17.651015
12,2022-01,-9.461070,0.9370,6.0,-8.714072
26,2023-03,4.110455,-3.4065,6.0,5.538719
38,2024-03,-1.711916,1.3370,6.0,3.514298
55,2025-08,0.327219,-2.2025,6.0,3.341238
58,2025-11,1.709330,-0.1775,6.0,4.203736
59,2025-12,2.159691,-0.4725,6.0,5.422422


,month,housing_credit_real_mom_pct,housing_rate_change_pp,baseline_observations,rolling_median,rolling_mad,robust_z,scorable,anomaly_flag
11,2021-12,-9.762532,0.2105,6.0,-1.194765,0.327396,-17.651015,True,True
12,2022-01,-9.461070,0.9370,6.0,-1.365367,0.626627,-8.714072,True,True
26,2023-03,4.110455,-3.4065,6.0,-2.579641,0.814703,5.538719,True,True
59,2025-12,2.159691,-0.4725,6.0,0.155858,0.249255,5.422422,True,True
58,2025-11,1.709330,-0.1775,6.0,0.155858,0.249255,4.203736,True,True
38,2024-03,-1.711916,1.3370,6.0,-3.972182,0.433808,3.514298,True,True
55,2025-08,0.327219,-2.2025,6.0,-0.038186,0.073764,3.341238,True,True
60,2026-01,-2.878626,-1.0520,6.0,0.356826,0.780248,-2.796909,True,False
7,2021-08,-0.334865,-0.0715,6.0,-1.212373,0.213039,2.778224,True,False
46,2024-11,0.990959,-0.2815,6.0,-1.348000,0.642238,2.456420,True,False


## Çıktının yorumu
`anomaly_flag=True` bir ayın önceki gözlemlere göre olağandışı olduğunu gösterir. Mevcut ay kendi referans penceresine dahil edilmez. Yetersiz geçmiş, sıfır MAD veya eksik değer varsa skor null ve `scorable=False` olur; false anomali bayrağı bu satırların normal olduğunu kanıtlamaz. Kaynak revizyonu, kapsam farkı veya ekonomik hareket ayrı araştırılmalıdır. Bu bayraklar kalıcı değişim noktası sonucu olarak sunulmaz.

## 11. AI olmadan doğrudan SQL sorgulama
Bu bölüm, agent olmadan bir veri uzmanının lakehouse’u doğrudan kullanabildiğini gösterir. Sorgu DuckDB içinde çalışır; sonuçlar doğrudan tablolar ve katalog üzerinden alınır.

In [17]:
direct_query = con.sql("""
    SELECT month,
           round(bddk_housing_credit_stock_million_tl, 2) AS housing_stock_million_tl,
           round(TP_KTF12, 2) AS housing_rate_pct,
           round(housing_credit_real_mom_pct, 2) AS real_stock_mom_pct
    FROM analysis.housing_credit_monthly
    WHERE month BETWEEN '2025-01' AND '2026-06'
    ORDER BY month
""").df()
print(f'Doğrudan sorgu sonucu: {len(direct_query)} satır')
display(direct_query)
print('Bu sorgu için AI gerekmez; kullanıcı veya veri uzmanı tablo/kolon adlarını bilerek SQL çalıştırabilir.')

Doğrudan sorgu sonucu: 18 satır

,month,housing_stock_million_tl,housing_rate_pct,real_stock_mom_pct
0,2025-01,523455,40.54,-2.71
1,2025-02,535029,39.97,-0.06
2,2025-03,548385,39.38,0.04
3,2025-04,561793,39.56,-0.54
4,2025-05,575453,42.14,0.89
5,2025-06,583265,43.47,-0.02
6,2025-07,594607,42.56,-0.11
7,2025-08,608694,40.36,0.33
8,2025-09,623765,38.80,-0.73
9,2025-10,642144,37.94,0.39


Bu sorgu için AI gerekmez; kullanıcı veya veri uzmanı tablo/kolon adlarını bilerek SQL çalıştırabilir.


## 12. Veri hizalama ve frekans uyumu
Aynı analiz dönemlerinin ulusal aylık, bölgesel çeyreklik ve EVDS panellerinde nasıl temsil edildiğini kontrol ediyoruz. Çeyreklik gözlemler ara aylara kopyalanmamalıdır.

In [18]:
alignment_checks = []
# Expected sizes come from the declared competition window and province dimension.
expected_months = len(pd.period_range('2021-01', '2026-06', freq='M'))
expected_quarters = len(pd.period_range('2021Q1', '2026Q2', freq='Q'))
province_count = con.sql('SELECT count(DISTINCT province_key) FROM regional.province_dimension').fetchone()[0]
checks = [
    ('analysis monthly', "SELECT count(*) FROM analysis.housing_credit_monthly", expected_months),
    ('analysis quarterly', "SELECT count(*) FROM analysis.housing_credit_quarterly", expected_quarters),
    ('regional province-quarter', "SELECT count(*) FROM regional.housing_quarterly", province_count * expected_quarters),
    ('regional distinct province-quarter', "SELECT count(DISTINCT province_key || ':' || quarter) FROM regional.housing_quarterly", province_count * expected_quarters),
]
for name, query, expected in checks:
    actual = con.sql(query).fetchone()[0]
    alignment_checks.append({'check': name, 'actual': actual, 'expected': expected, 'status': 'PASS' if actual == expected else 'CHECK'})
display(pd.DataFrame(alignment_checks))

quarterly_codes = con.sql("SELECT series_code FROM evds.series_catalog WHERE frequency = 'ÜÇ AYLIK'").df()['series_code']
quarterly_columns = {code.replace('.', '_').replace('-', '_') for code in quarterly_codes}
frequency_checks = []
for table in sorted(name for name in tables.qualified_name if name.startswith('evds.') and name.endswith('_monthly_panel')):
    schema_name, table_name = table.split('.')
    available = set(columns.loc[columns.table_schema.eq(schema_name) & columns.table_name.eq(table_name), 'column_name'])
    selected = sorted(available & quarterly_columns)
    if not selected:
        continue
    non_null_cells = ' + '.join('count("' + col.replace('"', '""') + '")' for col in selected)
    # Count actual values in quarterly-series columns, not monthly panel rows.
    actual = con.sql(f"""
        SELECT {non_null_cells} FROM {table}
        WHERE CAST(right(target_period, 2) AS INTEGER) NOT IN (3, 6, 9, 12)
    """).fetchone()[0]
    frequency_checks.append({'table': table, 'quarterly_series_columns': len(selected), 'intermediate_month_non_null_cells': actual, 'status': 'PASS' if actual == 0 else 'CHECK'})
display(pd.DataFrame(frequency_checks))
print('Kontrol çeyreklik kaynak sütunlarının ara aylardaki gerçek dolu hücrelerini tarar; diğer frekans dönüşümleri alignment audit tablolarında ayrıca incelenir.')

,check,actual,expected,status
0,analysis monthly,66,66,PASS
1,analysis quarterly,22,22,PASS
2,regional province-quarter,1782,1782,PASS
3,regional distinct province-quarter,1782,1782,PASS


,table,quarterly_series_columns,intermediate_month_non_null_cells,status
0,evds.housing_controls_monthly_panel,6,0,PASS
1,evds.housing_monthly_panel,24,0,PASS
2,evds.regional_housing_monthly_panel,159,0,PASS


Kontrol çeyreklik kaynak sütunlarının ara aylardaki gerçek dolu hücrelerini tarar; diğer frekans dönüşümleri alignment audit tablolarında ayrıca incelenir.


## 13. BDDK kümülatif verilerinin ayrıştırılması
BDDK aylık semantik katmanında bazı kaynak değerler yıl içinde kümülatif ilerler. Builder bu değerleri aynı takvim yılı içinde farklayarak aylık akışa çevirir; Ocak ayı doğrudan kaynak değeri olarak korunur.

In [19]:
cumulative_result = con.sql("""
    SELECT
        count(*) AS cumulative_rows,
        count(*) FILTER (WHERE month LIKE '%-01') AS january_rows,
        max(abs(source_value - recomposed_value)) AS max_recomposition_difference
    FROM (
        SELECT table_no, metric_code, group_code, calendar_year, month,
               source_value,
               sum(analysis_value) OVER (
                   PARTITION BY table_no, metric_code, group_code, calendar_year
                   ORDER BY month
                   ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
               ) AS recomposed_value
        FROM bddk.monthly_measurements
        WHERE transformation = 'difference_within_calendar_year'
    )
""").df()
display(cumulative_result)
validation_path = DB_PATH.parents[1] / 'bddk' / 'processed' / 'monthly_semantic' / 'validation.json'
if validation_path.exists():
    validation = pd.read_json(validation_path, typ='series')
    print('Builder validation özeti:')
    print(validation.get('recomposition', {}))
print('Beklenen sonuç: max_recomposition_difference sıfıra yakın olmalı; bu, aylık akışların kaynak kümülatif değerleri yeniden oluşturduğunu gösterir.')

,cumulative_rows,january_rows,max_recomposition_difference
0,104940,9540,0.0


Builder validation özeti:
{'cumulative_measurement_rows': 104940, 'recomposition_comparisons': 104940, 'maximum_recomposition_difference': 0.0, 'january_start_rows': 9540, 'annual_reset_pairs_checked': 7950, 'annual_reset_pairs_with_lower_january_absolute_value': 7069, 'annual_reset_share': 0.8891819999999999}
Beklenen sonuç: max_recomposition_difference sıfıra yakın olmalı; bu, aylık akışların kaynak kümülatif değerleri yeniden oluşturduğunu gösterir.


## 14. Veri kalitesi ve ilişkilendirilebilirlik özeti
Bu bölüm dönem anahtarlarını, TÜİK-EVDS değer uyuşmazlıklarını ve açık TBB kaynak boşluklarını özetler. Önceki ilişki tablosu eşleşmeyen anahtarları ayrıca gösterir. Bu teknik kontroller ekonomik ölçü tanımlarının doğru seçildiğini tek başına kanıtlamaz.

In [20]:
quality_summary = []
quality_summary.append({'check': 'monthly duplicate periods', 'value': con.sql("SELECT count(*) - count(DISTINCT month) FROM analysis.housing_credit_monthly").fetchone()[0], 'expected': 0})
quality_summary.append({'check': 'quarterly duplicate periods', 'value': con.sql("SELECT count(*) - count(DISTINCT quarter) FROM analysis.housing_credit_quarterly").fetchone()[0], 'expected': 0})
if has_table('tuik.province_housing_sales_evds_reconciliation'):
    mismatch = con.sql("SELECT count(*) FROM tuik.province_housing_sales_evds_reconciliation WHERE reconciliation_status = 'value_mismatch'").fetchone()[0]
    quality_summary.append({'check': 'TÜİK-EVDS value mismatches', 'value': mismatch, 'expected': 0})
if has_table('tbb.source_gaps'):
    gaps = con.sql("SELECT count(*) FROM tbb.source_gaps").fetchone()[0]
    quality_summary.append({'check': 'declared TBB source gaps', 'value': gaps, 'expected': 'explicitly documented'})
quality_df = pd.DataFrame(quality_summary)
quality_df['status'] = quality_df.apply(lambda row: 'PASS' if row['expected'] == 0 and row['value'] == 0 else ('INFO' if row['expected'] != 0 else 'CHECK'), axis=1)
display(quality_df)
print('PASS teknik tutarlılığı; INFO/CHECK ise veri kaynağında açıklanmış kapsam boşluklarını veya incelenmesi gereken durumları gösterir.')

,check,value,expected,status
0,monthly duplicate periods,0,0,PASS
1,quarterly duplicate periods,0,0,PASS
2,TÜİK-EVDS value mismatches,0,0,PASS
3,declared TBB source gaps,1,explicitly documented,INFO


PASS teknik tutarlılığı; INFO/CHECK ise veri kaynağında açıklanmış kapsam boşluklarını veya incelenmesi gereken durumları gösterir.


## 15. Ortak katalog alanlarının incelenmesi
`catalog.metrics`, `catalog.data_assets` ve `catalog.table_manifest` kaynak, frekans, birim, gözlem durumu ve dosya kökenini ortak alanlarla taşır. Sorgular belirli bir ekonomik metrik adına bağlı değildir. Bunun görülmesi, bilinmeyen bir veri kaynağının otomatik hazırlanabildiğini veya genel agent'ın tamamlandığını kanıtlamaz.

In [21]:
generic_metrics = con.sql("""
    SELECT source_system,
           native_frequency,
           unit,
           count(*) AS metric_count,
           count(*) FILTER (WHERE observation_available) AS locally_queryable_count
    FROM catalog.metrics
    GROUP BY source_system, native_frequency, unit
    ORDER BY source_system, native_frequency, unit
""").df()
display(generic_metrics)

generic_assets = con.sql("""
    SELECT source_system,
           data_kind,
           native_frequency,
           count(*) AS asset_count,
           sum(row_count) AS total_rows
    FROM catalog.data_assets
    GROUP BY source_system, data_kind, native_frequency
    ORDER BY source_system, data_kind, native_frequency
""").df()
display(generic_assets)

manifest = con.sql("""
    SELECT schema_name, table_name, row_count, source_path
    FROM catalog.table_manifest
    ORDER BY schema_name, table_name
""").df()
display(manifest.head(30))
print('Ortak katalog alanları kaynakları keşfetmeye yardımcı olur; yeni kaynağın ingestion ve semantik doğrulaması ayrıca gereklidir.')

,source_system,native_frequency,unit,metric_count,locally_queryable_count
0,BDDK_FINTURK,quarterly,count,1,1
1,BDDK_FINTURK,quarterly,people_per_branch,1,1
2,BDDK_FINTURK,quarterly,percent,16,16
3,BDDK_FINTURK,quarterly,thousand_try,54,54
4,BDDK_FINTURK,quarterly,try_per_person,4,4
...,...,...,...,...,...
89,TCMB_EVDS,İŞ GÜNÜ,milyon ABD doları,38,0
90,TCMB_EVDS,İŞ GÜNÜ,milyon TL,2,0
91,TCMB_EVDS,İŞ GÜNÜ,milyon TL ve yüzde,12,1
92,TCMB_EVDS_DERIVED,derived,TL/gram,1,1


,source_system,data_kind,native_frequency,asset_count,total_rows
0,BDDK_FINTURK,identity_derived_zero_audit,quarterly,1,1328.0
1,BDDK_FINTURK,missingness_audit,quarterly,1,47.0
2,BDDK_FINTURK,observations_long,quarterly,1,936512.0
3,BDDK_FINTURK,source_table,quarterly,7,84484.0
4,BDDK_MONTHLY,semantic_observations_long,monthly,1,1334850.0
5,BDDK_MONTHLY,source_table,monthly,17,339650.0
6,BDDK_WEEKLY,missingness_audit,weekly,1,2.0
7,BDDK_WEEKLY,observations_long,weekly,1,1025974.0
8,BDDK_WEEKLY,raw_source_collection,weekly,1,18018.0
9,CROSS_SOURCE_QUALITY,cross_source_reconciliation,quarterly,1,22.0


,schema_name,table_name,row_count,source_path
0,analysis,housing_credit_monthly,66,"derived from validated BDDK, EVDS, Risk Center..."
1,analysis,housing_credit_quarterly,22,"derived from validated BDDK, FinTurk, EVDS and..."
2,bddk,finturk_branch_zero_fallback_audit,1328,data_pipeline/bddk/processed/finturk_all_group...
3,bddk,finturk_measurements,936512,data_pipeline/bddk/processed/finturk_all_group...
4,bddk,finturk_metric_dictionary,76,data_pipeline/bddk/processed/finturk_all_group...
5,bddk,finturk_missingness_audit,47,data_pipeline/bddk/processed/finturk_all_group...
6,bddk,finturk_source_tables,154,data_pipeline/bddk/processed/finturk_all_group...
7,bddk,housing_credit_monthly,66,data_pipeline/bddk/processed/housing_credit_mo...
8,bddk,monthly_measurements,1334850,data_pipeline/bddk/processed/monthly_semantic/...
9,bddk,monthly_metric_dictionary,2108,data_pipeline/bddk/processed/monthly_semantic/...


Ortak katalog alanları kaynakları keşfetmeye yardımcı olur; yeni kaynağın ingestion ve semantik doğrulaması ayrıca gereklidir.


## Generik yapı çıktısının yorumu
`catalog.metrics` ve `catalog.data_assets` içindeki ortak alanlar, yeni bir veri ailesinin aynı keşif ve lineage sözleşmesine eklenebildiğini gösterir. Ancak bu henüz tamamen otomatik bir agent sistemi olduğu anlamına gelmez: yeni kaynağın downloader’ı, normalizasyon kuralı ve kalite testleri ayrıca yazılmalıdır. Generic olan kısım kataloglama, sorgulanabilirlik ve metadata modelidir; kaynak özelindeki semantik dönüşümler yine ayrı kurallara ihtiyaç duyar.

# Analitik sınırlar ve sonraki adımlar

Kaynak dosyası, lineage ve teknik tutarlılık ekonomik yorumun ayrı parçalarıdır. Temiz checkout'ta dosya manifesti Git ile dağıtılan kapsamla uyumlu olmalıdır. Manifest doğrulaması ayrı test akışında yapılır; bu notebook veri indirmez veya dosya onarımı yapmaz.

Fiyat kullanılmayan analizlerde fiyat-hazırlık bayrağıyla örneklem daraltılmaz. Fiyat kullanan analizlerde resmî değer ve açık proxy seçimi ayrı tutulur; akran desteği mevcut veriden sayılır. Risk Merkezi, TBB ve BDDK kurum kapsamları, birimleri ve stok/akım anlamları korunur. Aşağıdaki iki yönlü model il ve çeyrek gösterge değişkenlerinin ortak doğrusal uzayına projeksiyon yapar; dengesiz panelde de sabit etkileri çıkarır. Standart hata, güven aralığı ve nedensel etki hesaplamaz.

## 1. Kaynak dönemlerini karşılaştırma
Bu hücre kaynak tablolarının fiilî başlangıç/bitiş dönemlerini gösterir. Aynı modelde kullanılacak değişkenler için ortak dolu gözlemler ayrıca seçilmelidir. Tablo genelindeki min/max dönemler her metriğin o dönemde dolu olduğunu veya gerçek yayın gününü göstermez.

In [22]:
source_coverage = con.sql("""
    SELECT 'BDDK aylık analiz' AS source, min(month) AS period_start, max(month) AS period_end, count(*) AS rows
    FROM analysis.housing_credit_monthly
    UNION ALL
    SELECT 'Risk Merkezi aylık', min(month), max(month), count(*)
    FROM risk_center.housing_credit_monthly
    UNION ALL
    SELECT 'TBB çeyreklik', min(report_period), max(report_period), count(*)
    FROM tbb.consumer_credit_product_metrics
    UNION ALL
    SELECT 'İl bazlı bölgesel panel', min(quarter), max(quarter), count(*)
    FROM regional.housing_quarterly
""").df()
display(source_coverage)
print('Kural: Modelde kullanılacak TBB kullandırım sütununun gerçek dolu dönemleriyle ortak örneklem seçilir; eksik yayın sıfır değildir.')

,source,period_start,period_end,rows
0,BDDK aylık analiz,2021-01,2026-06,66
1,Risk Merkezi aylık,2021-01,2026-06,66
2,TBB çeyreklik,2021-03,2026-03,252
3,İl bazlı bölgesel panel,2021Q1,2026Q2,1782


Kural: Modelde kullanılacak TBB kullandırım sütununun gerçek dolu dönemleriyle ortak örneklem seçilir; eksik yayın sıfır değildir.


### Çıktının analizi
Başlangıç, bitiş ve satır sayıları yukarıda güncel tablolardan hesaplanır. Aylık ve çeyreklik dönemler uygun frekans kuralıyla hizalanmalıdır. TBB'de yayımlanmayan kullandırım gözlemi sıfır, düşüş veya tahmin değildir. Referans ayı ile fiilî yayın tarihi farklı olabilir; bu tablo geçmişte hangi bilginin bilindiğini doğrulayan bir vintage analizi değildir.

## 2. Risk Merkezi ve BDDK kredi stokunu bağlam içinde karşılaştırma
Bu hücre iki seriyi aynı aylarda yan yana getirir. Tanımları ve raporlayan kurum kapsamı farklı olabileceği için fark, doğrulama bağlamıdır; seriler birbirinin yerine geçirilmez.

In [23]:
risk_vs_bddk = con.sql("""
    SELECT a.month,
           round(a.bddk_housing_credit_stock_million_tl, 2) AS bddk_stock_million_try,
           round(r.housing_credit_balance_billion_try * 1000, 2) AS risk_center_stock_million_try,
           round(100.0 * (r.housing_credit_balance_billion_try * 1000
             / NULLIF(a.bddk_housing_credit_stock_million_tl, 0) - 1), 2) AS difference_vs_bddk_pct,
           r.has_source_revision
    FROM analysis.housing_credit_monthly a
    JOIN risk_center.housing_credit_monthly r USING (month)
    ORDER BY a.month DESC
""").df()
display(risk_vs_bddk.head(12))
print('Not: fark; kapsam, yayın yöntemi veya yuvarlamadan kaynaklanabilir. Bu tablo nedensel karşılaştırma değildir.')

,month,bddk_stock_million_try,risk_center_stock_million_try,difference_vs_bddk_pct,has_source_revision
0,2026-06,801438,810000.0,1.07,False
1,2026-05,782918,792000.0,1.16,False
2,2026-04,767314,776000.0,1.13,False
3,2026-03,741972,751000.0,1.22,False
4,2026-02,715799,724000.0,1.15,False
5,2026-01,691343,700000.0,1.25,False
6,2025-12,678970,687000.0,1.18,False
7,2025-11,658770,666000.0,1.10,False
8,2025-10,642144,650000.0,1.22,False
9,2025-09,623765,632000.0,1.32,False


Not: fark; kapsam, yayın yöntemi veya yuvarlamadan kaynaklanabilir. Bu tablo nedensel karşılaştırma değildir.


### Çıktının analizi
Son gözlemlerin farkları güncel tablodan okunmalıdır; sabit bir fark aralığı varsayılmaz. Risk Merkezi'nin banka dışı kuruluşları da içerebilen kapsamı ve grafik yuvarlaması, BDDK ile aynı ölçü kabul edilmesini engeller. Farkın düzenli görünmesi sebebini tek başına ispatlamaz. `has_source_revision`, eldeki örtüşen yayınlar arasında gözlenmiş revizyonu işaretler; false olması hiç revizyon yapılmadığını veya aynı değerin geçmişte bilindiğini garanti etmez. Kaynaklar birbirinin eksik değerini doldurmaz.

## 3. Resmî fiyat ve proxy kalitesi
Bu hücre, resmî ve proxy satırlarını açıkça sayar. `peer_count`, proxy medyanına giren resmî il sayısıdır; düşük değerli proxy'ler sonuç yorumunda daha ihtiyatlı ele alınmalıdır.

In [24]:
proxy_quality = con.sql("""
    SELECT housing_unit_price_proxy_origin AS price_origin,
           count(*) AS province_quarter_rows,
           min(housing_unit_price_proxy_peer_count) AS min_peer_count,
           median(housing_unit_price_proxy_peer_count) AS median_peer_count,
           max(housing_unit_price_proxy_peer_count) AS max_peer_count
    FROM regional.housing_quarterly
    GROUP BY housing_unit_price_proxy_origin
    ORDER BY price_origin
""").df()
display(proxy_quality)
proxy_support = con.sql("""
    SELECT housing_unit_price_proxy_peer_count AS peer_count, count(*) AS proxy_rows
    FROM regional.housing_quarterly
    WHERE housing_unit_price_proxy_origin = 'same_region_same_quarter_official_median_proxy'
    GROUP BY housing_unit_price_proxy_peer_count
    ORDER BY peer_count
""").df()
display(proxy_support)

,price_origin,province_quarter_rows,min_peer_count,median_peer_count,max_peer_count
0,official_source,1620,<NA>,NaN,<NA>
1,same_region_same_quarter_official_median_proxy,162,2,5.0,8


,peer_count,proxy_rows
0,2,40
1,3,40
2,5,50
3,6,28
4,8,4


### Çıktının analizi
Resmî ve proxy satır sayıları, proxy akran dağılımı ve medyanı üstteki iki tablodan hesaplanır. Düşük akran desteği, proxy'nin daha az resmî il gözlemine dayandığını gösterir; kendi başına hata olasılığı veya güven aralığı vermez. Resmî sütun null kalır. Proxy kullanan her analizde değer kökeni ve destek sayısı görünür tutulmalıdır.

## 4. Proxy kullanımının örnekleme etkisi
Bu hücre önce yalnız resmî fiyat içeren örneklemi, sonra proxy'ye açık örneklemi sayar. Sonraki model sonuçları iki örneklem için ayrı sunulmalıdır; proxy satırları resmî gözlemmiş gibi birleştirilmez.

In [25]:
sample_comparison = con.sql("""
    SELECT 'yalnız_resmi_fiyat' AS sample,
           count(*) AS rows,
           count(DISTINCT province_key) AS provinces,
           count(DISTINCT quarter) AS quarters
    FROM regional.housing_quarterly
    WHERE analysis_ready_source
    UNION ALL
    SELECT 'proxy_izinli_fiyat', count(*), count(DISTINCT province_key), count(DISTINCT quarter)
    FROM regional.housing_quarterly
    WHERE analysis_ready_with_price_proxy
    UNION ALL
    SELECT 'proxy_ve_en_az_4_akran', count(*), count(DISTINCT province_key), count(DISTINCT quarter)
    FROM regional.housing_quarterly
    WHERE analysis_ready_with_price_proxy
      AND (housing_unit_price_proxy_origin = 'official_source'
           OR housing_unit_price_proxy_peer_count >= 4)
""").df()
display(sample_comparison)

,sample,rows,provinces,quarters
0,yalnız_resmi_fiyat,1620,76,22
1,proxy_izinli_fiyat,1782,81,22
2,proxy_ve_en_az_4_akran,1702,81,22


### Çıktının analizi
Tablo üç açık fiyat politikasında kalan gözlem, il ve çeyrek sayısını gösterir. En az dört akran eşiği bu keşif örneğinin duyarlılık tercihidir; resmî kalite standardı değildir. Bu hücre yalnız örneklem kapsamını karşılaştırır, üç ayrı model çalıştırmaz. Katsayıların proxy politikasına duyarlılığı konusunda sonuç vermek için aynı modelin bu örneklemlerde ayrıca kurulması gerekir.

## 5. İki yönlü sabit etkiyle keşifsel panel modeli
Bağımlı değişken `log(1 + satış)`tır. İl ve çeyrek gösterge değişkenleri, zaman içinde sabit il düzeyleri ile tüm illerin paylaştığı çeyrek düzeyi bileşenlerini modelden çıkarır. Resmî fiyat örneklemi dengesiz olabileceği için il ve dönem ortalamalarını tek sefer çıkarmak yerine ortak gösterge matrisine en küçük kareler projeksiyonu uygulanır. Tüm model değişkenleri aynı complete-case örneklemde dönüştürülür. Eşanlılık ve ölçülmeyen etkenler nedeniyle sonuçlar nedensel etki veya istatistiksel anlamlılık göstermez.

In [26]:
twfe = con.sql("""
    SELECT province_key, quarter,
           housing_sales_total_count AS sales_total,
           housing_unit_price_try_per_m2 AS official_price,
           housing_credit_per_capita_try AS credit_per_capita,
           mortgaged_sales_share_pct AS mortgaged_share
    FROM regional.housing_quarterly
""").df()
features = ['official_price', 'credit_per_capita', 'mortgaged_share']
twfe_source_rows = len(twfe)
# Only the official price column enters this model. Explicit price proxies
# remain outside this sample, while unrelated readiness flags do not filter it.
twfe = twfe.dropna(subset=['province_key', 'quarter', 'sales_total', *features]).copy()
if twfe.duplicated(['province_key', 'quarter']).any():
    raise ValueError('İki yönlü modelde il-çeyrek anahtarı tekil değil.')
if twfe['sales_total'].lt(0).any():
    raise ValueError('Satış sayısı negatif olamaz.')
twfe['log_sales'] = np.log1p(twfe['sales_total'])
residualized = residualize_fixed_effects(
    twfe, ['log_sales', *features], ['province_key', 'quarter']
)
X = residualized[features].to_numpy()
y = residualized['log_sales'].to_numpy()
coefficients, _, rank, _ = np.linalg.lstsq(X, y, rcond=None)
if rank < len(features):
    raise ValueError('İl ve çeyrek etkileri çıkarıldıktan sonra model değişkenleri bağımsız değil.')
twfe_result = pd.DataFrame({
    'feature': features,
    'coefficient_per_unit': coefficients,
    'comparison_change': [1000, 1000, 1],
    'unit': ['TL/m²', 'TL/kişi', 'yüzde puan'],
})
twfe_result['associated_log1p_sales_difference'] = (
    twfe_result['coefficient_per_unit'] * twfe_result['comparison_change']
)
display(twfe_result)
counts = twfe.groupby('province_key')['quarter'].nunique()
print(f'Örneklem: {len(twfe)} il-çeyrek | İl: {twfe.province_key.nunique()} | Çeyrek: {twfe.quarter.nunique()} | Kullanılan sütunlarda eksik: {twfe_source_rows - len(twfe)}')
print(f'İl başına gözlenen çeyrek: {counts.min()} - {counts.max()}')
print('Katsayılar aynı örneklemde hesaplandı. Standart hata, güven aralığı veya nedensel etki hesaplanmadı.')

,feature,coefficient_per_unit,comparison_change,unit,associated_log1p_sales_difference
0,official_price,-0.000005,1000,TL/m²,-0.004847
1,credit_per_capita,0.000028,1000,TL/kişi,0.027834
2,mortgaged_share,0.005960,1,yüzde puan,0.005960


Örneklem: 1620 il-çeyrek | İl: 76 | Çeyrek: 22 | Kullanılan sütunlarda eksik: 162
İl başına gözlenen çeyrek: 14 - 22
Katsayılar aynı örneklemde hesaplandı. Standart hata, güven aralığı veya nedensel etki hesaplanmadı.


### Çıktının analizi
Katsayı ve birim başına değişimler güncel model sonucundan hesaplanır. `comparison_change` fiyat ve kişi başı kredi için 1.000 birim, ipotekli pay için bir yüzde puandır; bunların `log(1 + satış)` ile koşullu birlikte hareketi son sütunda verilir. Bu değer doğrudan satış yüzdesi veya gerçekleşecek bir what-if sonucu değildir. Kredi, fiyat ve satış birlikte belirlenebilir; gecikmeler, güven aralıkları ve standart hatalar bu örnekte yoktur. Çıktı daha ileri inceleme için keşifsel bir ilişki özetidir.

## Beş bulgunun birlikte değerlendirilmesi

1. **Dönem:** Her modelin kullandığı sütunlarda ortak dolu dönem seçilir; yayımlanmayan kaynak değeri sıfır değildir.
2. **Çapraz kaynak:** BDDK, Risk Merkezi ve TBB aynı kurum kapsamı veya ölçü değildir. Farklar gösterilir; kaynaklar birbirinin eksik değerini doldurmaz.
3. **Örneklem:** Kullanılmayan bir fiyat alanı başka analizlerden il dışlama nedeni olmaz. Fiyat kullanıldığında resmî gözlem ve açık proxy ayrı politikalardır.
4. **Duyarlılık:** Fiyat politikası değişince kalan örneklem sayısı raporlanır. Model duyarlılığı hakkında konuşmak için modeller ayrıca karşılaştırılmalıdır; bu notebook o karşılaştırmayı tamamladığını iddia etmez.
5. **Modelleme:** İl/çeyrek gösterge matrisi dengesiz panelde doğru sabit etki projeksiyonu sağlar. Ancak katsayılar nedensel değildir. Rolling-MAD bayrakları da kalıcı değişim noktası tespiti değildir.